# Notebook 01 — SimpleMLP 784→8→4→2 on MNIST even/odd (digits 0, 1, 3, 4)

**Architecture:** SimpleMLP with hidden dims [8, 4], trained on MNIST digits 0, 1, 3, 4 with even/odd binary labels.

## Structure
0. **Imports & setup**
1. **Configuration**
2. **Backward Factor Trace** — seed-0 model, BFT, exploratory plots (pixel receptive fields, scaffold graphs)
3. **BFT figures** — main-paper figure 2 and its appendix companion
4. **Fingerprints** — NNLS factor tree, fingerprint similarity, near-OOD (excluded digits), far-OOD (synthetic)
5. **Fingerprint figures** — main paper and appendix (placeholders)

Validation, robustness and ablation analyses live in notebook 09.

## §0 — Imports & setup

In [1]:
#%matplotlib inline
import sys, os
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets
from torchvision.transforms import ToTensor

from src import (
    SimpleMLP, load_experiment, get_transform, get_loaders_from_config,
    collect_layer_inputs_generic, collect_layer_dicts, bft, evaluate, build_scaffold_edges,
    scaffold_loading_from_edges, plot_scaffold_graph, extract_tree_nodes,
    extract_factor_tree_nodes, extract_fingerprint_matrix, compute_stimulus_similarity,
    project_stimuli_onto_tree, compute_factor_activations, plot_factor_tree,
    plot_input_layer_factors, plot_embedding_comparison, save_experiment,
)
from src.training import train_epoch, label_transform_even_odd
from src.data_utils import get_mnist_loaders, label_transformed_loader

matplotlib.rcParams.update({'figure.dpi': 80})
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cpu


## §1 — Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
_nb_dir   = os.path.dirname(os.path.abspath('__file__'))
_repo_dir = os.path.dirname(_nb_dir)

MODEL_ROOT = os.path.join(_repo_dir, 'data', 'models')
FIG_DIR    = os.path.join(_repo_dir, 'figs', '01_mlp_8_4_0134')
for d in [MODEL_ROOT, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Experiment ────────────────────────────────────────────────────────────────
EXP_BASE     = 'mnist_even_odd_mlp_8_4_0134'
N_EPOCHS     = 30
DIGIT_FILTER = [0, 1, 3, 4]
N_CLASSES    = 2
CLASS_NAMES  = {0: 'even', 1: 'odd'}
IMAGE_SIDE   = 28

BASE_CONFIG = {
    'arch': 'SimpleMLP',
    'arch_kwargs': {'input_dim': 784, 'hidden_dims': [8, 4], 'output_dim': 2},
    'dataset': 'MNIST',
    'dataset_kwargs': {'root': '../data/', 'batch_size': 32,
                       'digit_filter': DIGIT_FILTER},
    'label_transform': 'even_odd',
    'analysis_layer_indices': [2, 4, 6],
    'n_per_class': 1000,
    'input_side': 28,
}

# ── BFT config — PUBLICATION SETTINGS (see PUBLICATION_SETTINGS.md) ───────────
# From the nb09 S10 sweep (`rank x0.7`, run 4). The sweep passed [4, 1, 1], but bft's
# _auto_k_factorize floors k_max at 2 (`max(int(k_max), 2)`), so 1 and 2 behave identically
# here -- verified bit-identical fingerprints. [4, 2, 2] is written instead because it is
# what actually executes.
#
# Net effect vs the old [5, 5, 5]: ONLY the input layer changes (rank 5 -> 4). The root and
# hidden layers already auto-selected K = 2 / 1 / 2 and are untouched. Measured:
#   silhouette  0.666 -> 0.711     fingerprint  15 -> 13 dims
#   median causal R2  0.968 -> 0.955   BUT worst-node R2  0.778 -> 0.838 (improves)
N_BRANCHES         = [1, 1, 2]

STIMULUS_THRESHOLD = 0.5
# Causal-validation sample count. The default (100) is what produced the spurious
# R2 = -0.484 input node; at 2000 the same node reconstructs at +0.778.
VALIDATE_TOP_M     = 2000

# ── Fingerprint / OOD config (§4) ─────────────────────────────────────────────
N_FAR_OOD = 300

print(f'MODEL_ROOT: {MODEL_ROOT}')
print(f'FIG_DIR:    {FIG_DIR}')

MODEL_ROOT: /Users/dyballa/python/Factor_Trace/data/models
FIG_DIR:    /Users/dyballa/python/Factor_Trace/figs/01_mlp_8_4_0134


## §2 — Backward Factor Trace (seed 0)

### 2a — Model, loaders and layer activations

In [3]:
# ── Seed-0 model: load or train ───────────────────────────────────────────────
exp_dir = os.path.join(MODEL_ROOT, f'{EXP_BASE}_seed0')

if os.path.exists(os.path.join(exp_dir, 'weights.pt')):
    model, config = load_experiment(exp_dir, DEVICE)
    print(f'Loaded model from {exp_dir}')
else:
    print('No checkpoint found, training from scratch...')
    torch.manual_seed(0)
    np.random.seed(0)
    model  = SimpleMLP(**BASE_CONFIG['arch_kwargs']).to(DEVICE)
    config = dict(BASE_CONFIG, description=f'{EXP_BASE} seed 0')

    _train_loader, _ = get_mnist_loaders(batch_size=32, root='../data/',
                                         digit_filter=DIGIT_FILTER)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    for ep in range(N_EPOCHS):
        train_epoch(model, _train_loader, optimizer, criterion, DEVICE,
                    label_transform_even_odd)
    save_experiment(model, config, exp_dir)
    print(f'Saved to {exp_dir}')

train_loader, test_loader = get_loaders_from_config(config)
label_transform = get_transform(config['label_transform'])
digit_filter    = config['dataset_kwargs']['digit_filter']
_, test_acc     = evaluate(model, test_loader, nn.CrossEntropyLoss(), DEVICE, label_transform)

linear_indices = model.linear_layer_indices()
LAYER_SIZES    = [model.layers[li].out_features for li in linear_indices]
print(f'Seed-0 test accuracy: {test_acc:.4f}')
print(f'Architecture (per-layer neurons): {LAYER_SIZES}')

Loaded model from /Users/dyballa/python/Factor_Trace/data/models/mnist_even_odd_mlp_8_4_0134_seed0
Seed-0 test accuracy: 0.9944
Architecture (per-layer neurons): [8, 4, 2]


In [4]:
from src.data_utils import label_transformed_loader

# Loader yielding even/odd labels, for BFT primary mode (enables validate=True).
val_loader = label_transformed_loader(test_loader, label_transform)

# Collect the SAME samples BFT primary mode uses (loader order, only_correct), so
# downstream per-sample arrays stay aligned with tree_root's img_factors.
_collected  = collect_layer_dicts(model, test_loader, label_transform=label_transform,
                                  device=DEVICE)
all_images   = _collected['images']                              # (N, 1, 28, 28)
all_targets  = _collected['targets']                             # (N,) even/odd labels
all_digits   = _collected['digits']                              # (N,) original digit
layer_inputs = [d['input_fmap'] for d in _collected['layer_data']]  # list[(N, n_in)]
n_samples    = len(all_images)
flat_imgs    = all_images.reshape(n_samples, -1)

print(f'Samples: {n_samples}')
print(f'Class distribution: {dict(zip(*np.unique(all_targets, return_counts=True)))}')
for i, li in enumerate(layer_inputs):
    print(f'  L{i+1} inputs: {li.shape}')

Samples: 4084
Class distribution: {np.int64(0): np.int64(1953), np.int64(1): np.int64(2131)}
  L1 inputs: (4084, 784)
  L2 inputs: (4084, 8)
  L3 inputs: (4084, 4)


### 2b — Run BFT

In [5]:
# ── NMF vs semi-NMF across K_MAX ──────────────────────────────────────────────
import time, itertools
import numpy as np
from collections import deque
from sklearn.metrics.pairwise import paired_cosine_distances as _pcd

def walk(root):
    out, q = [], deque([root])
    while q:
        n = q.popleft(); out.append(n); q.extend(n.children)
    return out

K_GRID = [[2, 2, 2],
          [3, 2, 2],        
          [4, 2, 2],        # published (Table 1)
          [6, 3, 2],
          [8, 4, 2],
          [12, 6, 2]]

METHODS = ['nmf', 'semi_nmf']

runs = {}
for method, kmax in itertools.product(METHODS, K_GRID):
    key = (method, tuple(kmax))
    t0 = time.time()
    r = bft(model, val_loader, k_max=kmax, n_branches=N_BRANCHES,
            stimulus_threshold=STIMULUS_THRESHOLD,
            factorization=method, verbose=0)
    runs[key] = dict(result=r, seconds=time.time() - t0)
    print(f'{method:9s} kmax={kmax}  {runs[key]["seconds"]:5.1f}s  '
          f'K*={[n.img_factors.shape[1] for n in r.nodes()]}')

/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


nmf       kmax=[2, 2, 2]   48.7s  K*=[2, 1, 2, 2, 2]


/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


nmf       kmax=[3, 2, 2]   59.6s  K*=[2, 1, 2, 3, 3]


/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/s

...

nmf       kmax=[6, 3, 2]   75.9s  K*=[2, 1, 2, 6, 6]


/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


nmf       kmax=[8, 4, 2]   77.8s  K*=[2, 1, 2, 8, 8]


/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/dyballa/python/Factor_Trace/.pixi/envs/dev/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:2309: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


nmf       kmax=[12, 6, 2]   87.7s  K*=[2, 1, 2, 12, 12]
semi_nmf  kmax=[2, 2, 2]    9.0s  K*=[2, 1, 2, 2, 2]
semi_nmf  kmax=[3, 2, 2]   20.4s  K*=[2, 1, 2, 3, 3]
semi_nmf  kmax=[4, 2, 2]   24.9s  K*=[2, 1, 2, 4, 4]
semi_nmf  kmax=[6, 3, 2]   37.1s  K*=[2, 1, 2, 6, 6]
semi_nmf  kmax=[8, 4, 2]   20.9s  K*=[2, 2, 2, 8, 8]
semi_nmf  kmax=[12, 6, 2]   38.8s  K*=[2, 2, 3, 12, 12]


In [7]:
# ── Conditioning diagnostic across the grid ───────────────────────────────────
# Question: when a factorization takes more components than another from the same
# arbor, are the extra ones genuine structure or cancelling directions?
#
# A non-negative basis cannot cancel: every rank-1 term adds, so each factor has
# to be a real part of the data. A signed basis can represent structure A as
# (A + B) - B, where the extra factor exists only to subtract the excess the
# others carry. Two symptoms of that:
#   - large |cosine| between factor pairs (they point against each other)
#   - large cond(H) (the basis is near-degenerate, so many different non-negative
#     coefficient vectors reconstruct the same arbor almost equally well)
# The second is what breaks the NNLS projection: fit and projection pick different
# near-equivalent solutions, which is what drove the round-trip to 0.886 at
# node (0,0) with K=4.

hdr = (f"{'method':10s} {'kmax':12s} {'L':>2s} {'path':8s} {'K':>3s} "
       f"{'lam_ratio':>10s} {'min_cos':>9s} {'max|cos|':>9s} {'cond':>10s}")
print(hdr)
print('-' * len(hdr))

diag = {}
for (method, kmax), run in runs.items():
    for node in run['result'].nodes():
        H = node.connection_factors                      # (n_features, K)
        K = H.shape[1]
        # Column-normalize so the cosines compare directions only.
        Hn = H / (np.linalg.norm(H, axis=0, keepdims=True) + 1e-12)
        G = Hn.T @ Hn                                    # (K, K) pairwise cosines
        off = G[~np.eye(K, dtype=bool)] if K > 1 else np.array([0.0])

        rec = dict(K=K,
                   lam_ratio=float(node.lambdas[0] / (node.lambdas[-1] + 1e-12)),
                   min_cos=float(off.min()),             # most anti-aligned pair
                   max_abs_cos=float(np.abs(off).max()), # most collinear pair
                   cond=float(np.linalg.cond(H)))
        diag[(method, str(kmax), node.layer_idx, str(node.path))] = rec

        print(f"{method:10s} {str(kmax):12s} {node.layer_idx:2d} {str(node.path):8s} "
              f"{K:3d} {rec['lam_ratio']:10.3f} {rec['min_cos']:9.3f} "
              f"{rec['max_abs_cos']:9.3f} {rec['cond']:10.2f}")
        

method     kmax          L path       K  lam_ratio   min_cos  max|cos|       cond
---------------------------------------------------------------------------------
nmf        (2, 2, 2)     2 ()         2      1.462     0.053     0.053       1.22
nmf        (2, 2, 2)     1 (0,)       1      1.000     0.000     0.000       1.00
nmf        (2, 2, 2)     1 (1,)       2      2.565     0.423     0.423       1.94
nmf        (2, 2, 2)     0 (0, 0)     2      1.465     0.506     0.506       1.81
nmf        (2, 2, 2)     0 (1, 0)     2      1.075     0.413     0.413       1.55
nmf        (3, 2, 2)     2 ()         2      1.462     0.053     0.053       1.22
nmf        (3, 2, 2)     1 (0,)       1      1.000     0.000     0.000       1.00
nmf        (3, 2, 2)     1 (1,)       2      2.565     0.423     0.423       1.94
nmf        (3, 2, 2)     0 (0, 0)     3      1.345     0.406     0.647       2.43
nmf        (3, 2, 2)     0 (1, 0)     3      1.391     0.313     0.407       1.79
nmf        (4, 2

* Comparison between NMF and semi-NMF

A high cond means the factors overlap enough that no single set of loadings is well determined — different combinations reconstruct the arbor about equally well, so the loadings stop being a meaningful description of how a stimulus is routed and the fit and its projection can disagree. NMF stays between 1.5 and 3.7 even when given 12 factors for an 8-unit layer, while semi-NMF reaches 292.

Because semi-NMF factors can be negative, they can subtract from each other, and the fit is
free to represent structure `A` as `(A + B) − B` — the extra factor exists only to cancel the
excess the others carry. This breaks the reading BFT depends on: a connection factor is
supposed to name the weights a circuit uses, so a factor carrying cancellation mass points at
connections the network never recruits. NMF can't do this, since every rank-1 term adds and
each factor has to be a real part of the arbor. That's visible in `min_cos`, which goes to
−0.82 for semi-NMF at `L1` and is never negative under NMF.

`max|cos|` catches a second failure, distinct from the cancellation above. At `L2` node `(0,)`,
semi-NMF takes `K=2` once `k_max ≥ 4` with the two factors 99% identical (0.991, then 0.998),
and `min_cos` stays positive there — so these aren't cancelling, they're duplicates: one
circuit found twice. That's what the spectral criterion is meant to prevent (App. C.2), and
it happens anyway. NMF holds at `K=1` at this node for every ceiling.


* Rank saturation

`K*` follows `k_max` exactly at `L1` (2→3→4→6→8→12), so the rank there isn't being selected —
it's just the ceiling, which happens when `k_recon` never gets the reconstruction error under
`ε_r = 0.4` and falls back to `k_max` by definition. Both methods do this despite different λ
spectra, so it's a property of the criterion rather than of either factorization, and it's
worth squaring with App. D.6's claim that `K*` is usually set by the spectral criterion.

**To do:** repeat this comparison on the other models (digit MLP, CIFAR-10 CNN, TinyViT,
SqueezeNet). Five nodes on the smallest MLP is a pilot, not a result — the digit MLP is the
natural next one, since its wider layers give the factorization more genuine structure to
find.

__draft for revised appendix__

\subsection{Comparison against Semi-NMF}

Factorizing the positive and negative parts of the arbor separately (\S2.2) costs
two factorizations per node. Semi-NMF \citep{ding2010convex} constrains only one
factor to be non-negative, so a single component could carry both the excitatory
and the inhibitory connections of a circuit, at half the cost. We implemented it
with the non-negativity on the stimulus loadings, leaving Eq.~4 and the backward
pass unchanged, and traced the $8\times4\times2$ even/odd MLP over rank bounds from
2 to 12 per layer. It runs 2--4$\times$ faster and the circuits still separate by
parity, but the factors it recovers are not the parts the method needs.

A signed basis can represent structure $A$ as $(A+B)-B$: one component overshoots
and another subtracts the excess. The reconstruction is unaffected, but a
connection factor is meant to name the weights a circuit uses, and a component
carrying cancellation mass names connections the network never recruits. The
minimum pairwise cosine between connection factors at $L_1$ reaches $-0.82$ under
semi-NMF; under NMF it cannot go below zero, since every rank-1 term adds and each
factor has to be a real part of the arbor. At $L_2$ the failure is duplication
rather than cancellation: semi-NMF splits one circuit into two components with
pairwise cosine $0.998$, which the spectral criterion of \S C.2 does not catch,
while NMF holds that node at $K=1$ at every rank bound. The condition number of the
connection factors summarizes both: $1.5$ to $3.7$ for NMF across the whole sweep,
up to $292$ for semi-NMF. Loadings this entangled are no longer well determined,
and the NNLS projection of \S C.4 disagrees with the fit that produced it, which
degrades the fingerprints and everything built from them.

The parts-based decomposition NMF provides \citep{lee1999learning} is therefore worth
the cost of splitting the signs. Its components have to be real parts of the arbor;
semi-NMF's overshoot and cancel, so what a component names is partly an artifact of
the fit. Semi-NMF is faster per node, but its components degrade as the rank bound
grows, while NMF's stay well-conditioned from $K=2$ to $K=12$. The rank bound can
therefore be set loosely under NMF, which is what makes the single fit and prune of
\S C.2 workable across a whole tree.